# 10N-AI: Asistente Inteligente de Machine Learning
### Proyecto I - Inteligencia Artificial | ITCR San Carlos
### Modelo 10: Predicción del Precio de Acciones del Índice S&P 500
**Estudiantes:** Carlos Arce & Joshua Picado  
**Profesor:** Ing. Efrén Jiménez Delgado


# Análisis del Problema

### Formulación del Problema
El índice **Standard & Poor's 500 (S&P 500)** agrupa a las 500 empresas públicas líderes del mercado bursátil estadounidense y representa el indicador más representativo de la economía corporativa global. La determinación y proyección del **precio de cierre (`close`)** de estas acciones resulta indispensable para gestores de fondos, robots de inversión (*robo-advisors*) e inversores individuales.

El objetivo de este modelo es predecir de forma cuantitativa el **precio de cierre de una acción del S&P 500 (`close`)** en función de sus parámetros intradiarios: precio de apertura (`open`), precio máximo registrado (`high`), precio mínimo (`low`) y volumen negociado (`volume`).

Este modelo funcionará como una habilidad de inteligencia bursátil para el asistente inteligente **10N-AI**, permitiendo consultas como:
*"10N-AI, proyectá el precio de cierre para una acción del S&P 500 que abrió en $175.50, con máximo en $178.20 y mínimo en $174.80"*.

### Selección del Modelo Visto en Clase
Para este problema de **aprendizaje supervisado de regresión continua**, se selecciona **Random Forest Regressor (Bosque Aleatorio)** como modelo principal, y **Regresión Ridge** como modelo base de referencia.

#### Justificación de la Selección:
1. **Generalización Multi-Activo:** Al entrenar sobre una muestra representativa y diversificada de las 500 compañías del índice, el ensamble de árboles aprende a generalizar patrones de volatilidad y formaciones de velas tanto para compañías tecnológicas (*mega-cap*) como para empresas industriales y de consumo.
2. **Resistencia a Shocks y Volatilidad:** El ensamble combina múltiples árboles de decisión entrenados sobre submuestras aleatorias, suavizando fluctuaciones atípicas provocadas por eventos corporativos o publicación de balances trimestrales.
3. **Escalabilidad y Ligereza:** Permite generar un modelo compacto (< 2 MB) capaz de evaluar cientos de cotizaciones por segundo en la API REST de 10N-AI.


In [3]:
# @title Importación de Librerías y Configuración Global { display-mode: "form" }
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
print("Librerías importadas exitosamente para 10N-AI - Modelo 10.")


Librerías importadas exitosamente para 10N-AI - Modelo 10.


In [4]:
# @title Conexión a Google Drive y Carga del Dataset { display-mode: "form" }
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_dataset = '/content/drive/MyDrive/IA/Datasets/all_stocks_5yr.csv'
except Exception:
    ruta_dataset = 'Datasets/all_stocks_5yr.csv'

if not os.path.exists(ruta_dataset):
    for alt in ['../Datasets/all_stocks_5yr.csv', 'all_stocks_5yr.csv', '/content/all_stocks_5yr.csv']:
        if os.path.exists(alt):
            ruta_dataset = alt
            break

print(f"Cargando dataset desde: {ruta_dataset}")
df = pd.read_csv(ruta_dataset).dropna()
print(f"Dataset cargado con éxito. Total de registros bursátiles: {df.shape[0]:,} filas.")
display(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando dataset desde: /content/drive/MyDrive/IA/Datasets/all_stocks_5yr.csv
Dataset cargado con éxito. Total de registros bursátiles: 619,029 filas.


,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL


# Entendimiento de los Datos

### Diccionario de Variables
* **`date` (Temporal):** Fecha de la sesión de operaciones bursátiles (2013-2018).
* **`open` (Numérica continua):** Precio de apertura de la acción (USD).
* **`high` (Numérica continua):** Cotización máxima alcanzada en el día (USD).
* **`low` (Numérica continua):** Cotización mínima registrada en la sesión (USD).
* **`close` (Numérica continua - Target):** Precio de liquidación / cierre oficial de la acción (USD).
* **`volume` (Numérica continua):** Cantidad de acciones comercializadas en la jornada.
* **`Name` (Nominal):** Ticker o símbolo bursátil de la compañía (e.g. AAPL, MSFT, AMZN, JNJ).


In [5]:
# @title Inspección de Estructura, Estadísticas y Valores Nulos { display-mode: "form" }
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(df.info())
print("\n" + "="*60 + "\n")

print("=== CONTEO DE VALORES NULOS ===")
print(df.isnull().sum())
print("\n" + "="*60 + "\n")

print("=== TOP 10 ACCIONES CON MAYOR NÚMERO DE SESIONES ===")
print(df['Name'].value_counts().head(10))


=== INFORMACIÓN GENERAL DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
Index: 619029 entries, 0 to 619039
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    619029 non-null  object 
 1   open    619029 non-null  float64
 2   high    619029 non-null  float64
 3   low     619029 non-null  float64
 4   close   619029 non-null  float64
 5   volume  619029 non-null  int64  
 6   Name    619029 non-null  object 
dtypes: float64(4), int64(1), object(2)
memory usage: 37.8+ MB
None


=== CONTEO DE VALORES NULOS ===
date      0
open      0
high      0
low       0
close     0
volume    0
Name      0
dtype: int64


=== TOP 10 ACCIONES CON MAYOR NÚMERO DE SESIONES ===
Name
ZTS     1259
AAL     1259
AAPL    1259
AAP     1259
ABBV    1259
ABC     1259
ABT     1259
ACN     1259
ADBE    1259
WM      1259
Name: count, dtype: int64


### Hallazgos Principales del Entendimiento de Datos
1. **Volumen de Información:** Con más de 619,000 registros, el dataset abarca 5 años de negociación bursátil diaria de las empresas más influyentes del planeta.
2. **Dispersión de Cotizaciones:** Las cotizaciones abarcan desde acciones de pequeña capitalización (~$5 USD) hasta gigantes bursátiles superiores a $1,000 USD.
3. **Completitud Post-Saneamiento:** Al remover filas residuales con valores faltantes, se obtiene una base íntegra y de alta calidad estadística.


# Exploración de los Datos

Visualizaciones interactivas con **Plotly** analizando la dinámica del S&P 500.


In [6]:
# @title Distribución Interactiva de Precios de Cierre { display-mode: "form" }
sample_eda = df.sample(n=10000, random_state=42)

fig_hist = px.histogram(
    sample_eda,
    x='close',
    nbins=40,
    marginal='box',
    title='<b>Distribución de Precios de Cierre en el S&P 500 (Muestra 10k)</b>',
    labels={'close': 'Precio de Cierre (USD)'},
    color_discrete_sequence=['#6366f1']
)
fig_hist.update_layout(template='plotly_white')
fig_hist.show()

print("--- ANÁLISIS: DISTRIBUCIÓN DE PRECIOS ---")
print("• Alta concentración en el rango de $30 a $120 USD, correspondiente a la mediana típica de acciones en Wall Street.")
print("• Cola larga positiva impulsada por firmas tecnológicas de alta cotización unitaria.")


--- ANÁLISIS: DISTRIBUCIÓN DE PRECIOS ---
• Alta concentración en el rango de $30 a $120 USD, correspondiente a la mediana típica de acciones en Wall Street.
• Cola larga positiva impulsada por firmas tecnológicas de alta cotización unitaria.


In [7]:
# @title Relación Bivariada: Rango Intradiario vs Volumen { display-mode: "form" }
sample_eda['Spread'] = sample_eda['high'] - sample_eda['low']

fig_scat = px.scatter(
    sample_eda.head(2000),
    x='Spread',
    y='close',
    color='volume',
    hover_data=['Name', 'open'],
    title='<b>Rango Intradiario (High - Low) vs. Precio de Cierre</b>',
    labels={'Spread': 'Amplitud High - Low (USD)', 'close': 'Precio de Cierre (USD)'},
    color_continuous_scale='Viridis'
)
fig_scat.update_layout(template='plotly_white')
fig_scat.show()

print("--- ANÁLISIS: RANGO INTRADIARIO ---")
print("• Relación proporcional entre la amplitud de precios y el valor base de la acción.")


--- ANÁLISIS: RANGO INTRADIARIO ---
• Relación proporcional entre la amplitud de precios y el valor base de la acción.


In [8]:
# @title Matriz de Correlación Interactiva de Métricas Bursátiles { display-mode: "form" }
cols_corr = ['close', 'open', 'high', 'low', 'volume']
matriz_corr = sample_eda[cols_corr].corr()

fig_heatmap = px.imshow(
    matriz_corr,
    text_auto='.3f',
    color_continuous_scale='Blues',
    zmin=0.0,
    zmax=1.0,
    title='<b>Matriz de Correlación Interactiva (S&P 500)</b>'
)
fig_heatmap.update_layout(template='plotly_white', width=800, height=650)
fig_heatmap.show()

print("--- ANÁLISIS DE CORRELACIÓN ---")
print("• High y Low exhiben correlación > 0.999 con el precio de cierre, fijando la trayectoria de liquidación.")


--- ANÁLISIS DE CORRELACIÓN ---
• High y Low exhiben correlación > 0.999 con el precio de cierre, fijando la trayectoria de liquidación.


# Modelo de machine learning

### Ingeniería de Características y Preparación
1. **Submuestreo Estratificado:** Muestra balanceada de 25,000 registros multiactivo para optimización de hiperparámetros y serialización ágil.
2. **Variables Predictoras:** `open`, `high`, `low`, `volume`.
3. **Partición Train / Test:** 80% entrenamiento y 20% prueba.


In [9]:
# @title Partición Train / Test y Preparación de Datos { display-mode: "form" }
sample_df = df.sample(n=25000, random_state=42).reset_index(drop=True)

X = sample_df[['open', 'high', 'low', 'volume']]
y = sample_df['close']
columnas_predictoras = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Dimensiones de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de prueba (X_test): {X_test.shape}")
print(f"Columnas predictoras ({len(columnas_predictoras)}):", columnas_predictoras)


Dimensiones de entrenamiento (X_train): (20000, 4)
Dimensiones de prueba (X_test): (5000, 4)
Columnas predictoras (4): ['open', 'high', 'low', 'volume']


In [10]:
# @title Búsqueda de Hiperparámetros Óptimos (GridSearchCV) { display-mode: "form" }
espacio_hiperparametros = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 14, None],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=espacio_hiperparametros,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===")
print(f"Mejor puntuación R² en validación cruzada: {grid_rf.best_score_:.4f}")
print("Mejores hiperparámetros encontrados:", grid_rf.best_params_)


=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===
Mejor puntuación R² en validación cruzada: 0.9997
Mejores hiperparámetros encontrados: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}


In [11]:
# @title Entrenamiento del Modelo Final y Comparación con Modelo Base { display-mode: "form" }
modelo_rf = grid_rf.best_estimator_

# Modelo Base (Regresión Ridge)
modelo_base = Ridge(alpha=1.0)
modelo_base.fit(X_train, y_train)

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_rf = cross_val_score(modelo_rf, X_train, y_train, cv=cv_kfold, scoring='r2')
scores_base = cross_val_score(modelo_base, X_train, y_train, cv=cv_kfold, scoring='r2')

print("=== VALIDACIÓN CRUZADA (5-FOLD CV) ===")
print(f"• Random Forest Regressor - R² Promedio: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})")
print(f"• Modelo Base Ridge       - R² Promedio: {scores_base.mean():.4f} (+/- {scores_base.std():.4f})")


=== VALIDACIÓN CRUZADA (5-FOLD CV) ===
• Random Forest Regressor - R² Promedio: 0.9997 (+/- 0.0002)
• Modelo Base Ridge       - R² Promedio: 1.0000 (+/- 0.0000)


In [12]:
# @title Importancia Relativa de Variables Bursátiles (Gini) { display-mode: "form" }
df_importancias = pd.DataFrame({
    'Variable': columnas_predictoras,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

fig_imp = px.bar(
    df_importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='<b>Importancia de Variables en la Predicción del S&P 500 (Gini)</b>',
    labels={'Importancia': 'Importancia (Gini)', 'Variable': 'Métrica Bursátil'},
    color_discrete_sequence=['#6366f1']
)
fig_imp.update_layout(template='plotly_white')
fig_imp.show()

print("--- ANÁLISIS DE IMPORTANCIA ---")
print("• High y Low dominan la convergencia con más del 85% de la ponderación.")


--- ANÁLISIS DE IMPORTANCIA ---
• High y Low dominan la convergencia con más del 85% de la ponderación.


# Evaluación


In [13]:
# @title Métricas de Evaluación en Conjunto de Prueba { display-mode: "form" }
y_pred_rf = modelo_rf.predict(X_test)
y_pred_base = modelo_base.predict(X_test)

# Métricas RF
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

# Métricas Base
r2_base = r2_score(y_test, y_pred_base)
mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))

comparativa = pd.DataFrame({
    'Métrica': ['R² Score (Bondad de Ajuste)', 'MAE (Error Absoluto Medio USD)', 'RMSE (Raíz ECM USD)'],
    'Random Forest (Propuesto)': [f"{r2_rf:.4f}", f"${mae_rf:.2f}", f"${rmse_rf:.2f}"],
    'Regresión Ridge (Base)': [f"{r2_base:.4f}", f"${mae_base:.2f}", f"${rmse_base:.2f}"]
})

display(comparativa)

print("--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---")
print(f"• Random Forest alcanza un R² de {r2_rf*100:.2f}% con un error absoluto medio de solo ${mae_rf:.2f} USD.")


,Métrica,Random Forest (Propuesto),Regresión Ridge (Base)
0,R² Score (Bondad de Ajuste),0.9998,1.0000
1,MAE (Error Absoluto Medio USD),$0.45,$0.33
2,RMSE (Raíz ECM USD),$1.36,$0.65


--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---
• Random Forest alcanza un R² de 99.98% con un error absoluto medio de solo $0.45 USD.


In [14]:
# @title Gráfico Interactivo de Valores Reales vs. Predichos { display-mode: "form" }
df_eval = pd.DataFrame({
    'Real': y_test,
    'Predicho': y_pred_rf,
    'Residuo': y_test - y_pred_rf
})

fig_pred = px.scatter(
    df_eval,
    x='Real',
    y='Predicho',
    hover_data=['Residuo'],
    title='<b>Precio Real vs. Predicho en S&P 500 (Random Forest)</b>',
    labels={'Real': 'Precio Real (USD)', 'Predicho': 'Precio Predicho por 10N-AI (USD)'},
    color_discrete_sequence=['#10b981']
)

min_val = min(y_test.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_rf.max())

fig_pred.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Ajuste Perfecto (y = x)',
    line=dict(color='#ef4444', dash='dash', width=2)
))

fig_pred.update_layout(template='plotly_white')
fig_pred.show()


In [15]:
# @title Distribución Interactiva de Residuos { display-mode: "form" }
fig_res = px.histogram(
    df_eval,
    x='Residuo',
    nbins=25,
    marginal='box',
    title='<b>Distribución de Errores Residuales (S&P 500)</b>',
    labels={'Residuo': 'Error Residual (USD)'},
    color_discrete_sequence=['#6366f1']
)
fig_res.update_layout(template='plotly_white')
fig_res.show()

print(f"• Media de Residuos: ${df_eval['Residuo'].mean():.2f}.")


• Media de Residuos: $-0.01.


# Resultados

### Simulador Interactivo y Exportación para el Asistente 10N-AI
La siguiente celda define la función interactiva para cotizar acciones desde **10N-AI** y genera el artefacto binario `.joblib` que será consumido de forma inmediata por la API REST.


In [16]:
# @title Simulador Interactivo de Cotización de Acciones para 10N-AI { display-mode: "form" }
def cotizar_accion_sp500_10n(open_price=150.0, high_price=152.5, low_price=149.2, volume=12000000.0):
    """
    Función de inferencia consumible por el asistente 10N-AI.
    """
    fila = {
        'open': open_price,
        'high': high_price,
        'low': low_price,
        'volume': volume
    }
    entrada_df = pd.DataFrame([fila])[columnas_predictoras]
    precio_predicho = modelo_rf.predict(entrada_df)[0]

    spread = high_price - low_price
    print("="*65)
    print("🤖 [10N-AI ASISTENTE]: Estimación Bursátil de Acción S&P 500:")
    print(f"• Apertura: ${open_price:.2f} | Rango Sesión: [${low_price:.2f} - ${high_price:.2f}] | Spread: ${spread:.2f}")
    print(f"• Volumen Negociado: {volume:,.0f} títulos")
    print(f"📈 PRECIO DE CIERRE PROYECTADO: ${precio_predicho:.2f} USD")
    print("="*65)
    return precio_predicho

# Prueba de simulación
_ = cotizar_accion_sp500_10n(open_price=175.0, high_price=177.5, low_price=174.0, volume=8500000)


🤖 [10N-AI ASISTENTE]: Estimación Bursátil de Acción S&P 500:
• Apertura: $175.00 | Rango Sesión: [$174.00 - $177.50] | Spread: $3.50
• Volumen Negociado: 8,500,000 títulos
📈 PRECIO DE CIERRE PROYECTADO: $176.12 USD


In [17]:
# @title Exportación del Modelo Serializado para la API REST { display-mode: "form" }
import os
import joblib

try:
    from google.colab import files
    es_colab = True
except ImportError:
    es_colab = False

os.makedirs('/content/models', exist_ok=True)
ruta_export = '/content/models/modelo_10_sp500.joblib'
ruta_local = '10N-AI/models/modelo_10_sp500.joblib'

paquete_modelo = {
    'model': modelo_rf,
    'features': columnas_predictoras,
    'metrics': {
        'r2': float(r2_rf),
        'mae': float(mae_rf),
        'rmse': float(rmse_rf)
    },
    'target': 'close (USD)'
}

dest = ruta_export if os.path.exists('/content') else ruta_local
os.makedirs(os.path.dirname(dest), exist_ok=True)
joblib.dump(paquete_modelo, dest)
print(f"✅ Modelo exportado exitosamente a: {dest}")

if es_colab and os.path.exists(ruta_export):
    files.download(ruta_export)


✅ Modelo exportado exitosamente a: /content/models/modelo_10_sp500.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusiones del Análisis
1. **Fuga de Datos (Data Leakage) y Ajuste Lineal Perfecto:** En el conjunto de prueba, el modelo Random Forest logró un coeficiente $R^2$ de 0.9998 y un Error Absoluto Medio (MAE) de apenas 0.45 USD. Sin embargo, el modelo base de Regresión Ridge alcanzó un ajuste perfecto ($R^2$ de 1.0000 con un MAE de $0.33 USD). Este desempeño "perfecto" evidencia una severa Fuga de Datos en la formulación clínica del problema: al alimentar al algoritmo con los precios `high` y `low` de la misma jornada bursátil (datos que solo se conocen al cierre de la sesión), el modelo no realiza una proyección a futuro, sino una interpolación matemática lineal determinista. Como herramienta predictiva para el día de mañana, este enfoque es inválido; no obstante, funciona como un excelente motor de reconstrucción intradiaria.
2. **Jerarquía Bursátil de Gini:** El análisis de importancia de variables confirma el diagnóstico de fuga de datos. La cota superior `high` absorbe el 53.2% de la capacidad de decisión del árbol, mientras que la cota inferior `low` acapara el 32.6%. El precio de apertura (`open`) aporta un 14.1% dictando la inercia inicial, mientras que el volumen de transacciones (`volume`) resulta estadísticamente irrelevante (~0.01%). El algoritmo ignoró el peso del mercado y se limitó a triangular la posición del cierre basada en sus límites diarios.
3. **Factibilidad de Integración en 10N-AI:** A pesar de sus limitaciones como oráculo predictivo a futuro, el modelo es matemáticamente preciso para cálculos de sesión. Gracias al submuestreo de datos y la restricción de profundidad máxima a 10 niveles mediante GridSearchCV, el artefacto serializado resulta extremadamente ligero, garantizando latencias de inferencia de escasos milisegundos en la API REST del asistente.